In [ ]:
%%writefile check_semantic.py
import os
IS_SUB=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
import polars as pl
from scipy.special import softmax

# Configuration
N_PAIRS = 3  # Number of positive/negative example pairs (must match prepare script)

SYS_PRMPT = """
You are an experienced, fair, unbiased moderator. 
Classify whether the comment violates the supplied moderation rule.
- True: the comment breaks the specified rule
- False: the comment is considered safe in relation to the specified rule
Respond only using True or False.
""".strip()

def build_dynamic_prompt_template(n_pairs):
    """Build prompt template dynamically based on N_PAIRS"""
    template_parts = ["[RULE]: {}"]
    
    # Add True/False example pairs
    for i in range(n_pairs):
        template_parts.append(f"[True EXAMPLE {i+1}]: {{{i+1}}}")
        template_parts.append(f"[False EXAMPLE {i+1}]: {{{i+1+n_pairs}}}")
    
    template_parts.append("[TEST CASE COMMENT]: {" + str(2*n_pairs+1) + "}")
    
    return "\n".join(template_parts)

# Dynamic prompt template based on N_PAIRS
USR_PRMPT_TMPLT = build_dynamic_prompt_template(N_PAIRS)

CHOICES = ['True', 'False']

def chat_formatting(df, tokenizer):
  prompts = []
  for user_content in df['user_content']:
    chat = [
      {'role': 'system', 'content': SYS_PRMPT},
      {'role': 'user', 'content': user_content.strip()},
    ]
    prompt = tokenizer.apply_chat_template(
      chat, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    prompts.append(prompt)
  df = df.with_columns(pl.Series('prompt', prompts))
  return df


def prepare_user_content(df, n_pairs):
    """Prepare user content with dynamic example selection"""
    
    # Build the column list for format
    format_columns = ["rule"]
    
    # Add positive examples
    for i in range(n_pairs):
        format_columns.append(f"semantic_positive_example_{i+1}")
    
    # Add negative examples  
    for i in range(n_pairs):
        format_columns.append(f"semantic_negative_example_{i+1}")
    
    # Add target body
    format_columns.append("body")
    
    print(f"Using {n_pairs} example pairs:")
    print(f"Format columns: {format_columns}")
    
    # Apply dynamic formatting
    df = df.with_columns(
        pl.format(USR_PRMPT_TMPLT, *format_columns).alias('user_content')
    )
    
    return df


if __name__ == '__main__':
  print('importing vllm...')
  import vllm
  from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
  print('loading vllm...')

  llm = vllm.LLM(
    '/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1',
    quantization='awq',
    task='generate',
    tensor_parallel_size=2,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
    enable_prefix_caching=True,
    dtype='half',
    enforce_eager=True,
    disable_log_stats=True,
    disable_custom_all_reduce=True,
  )
  print('building prompts...')
  
  # Load pre-computed semantic examples
  path = 'test_with_semantic_examples_basicllm.csv'
  
  # Load and clean data
  df = pl.read_csv(path)
  
  # Clean whitespace in body and semantic example columns
  semantic_columns = []
  for i in range(N_PAIRS):
      semantic_columns.extend([
          f"semantic_positive_example_{i+1}",
          f"semantic_negative_example_{i+1}"
      ])
  
  clean_columns = ["body"] + semantic_columns
  df = df.with_columns([
      pl.col(col).str.replace_all(r'\s+', ' ') for col in clean_columns
  ])
  
  # Prepare user content dynamically
  df = prepare_user_content(df, N_PAIRS)

  tokenizer = llm.get_tokenizer()
  df = chat_formatting(df, tokenizer)
  prompts = df['prompt'].to_list()
  mclp = MultipleChoiceLogitsProcessor(
    tokenizer,
    choices=CHOICES,
  )
  sampling_params_choice = vllm.SamplingParams(seed=1337, skip_special_tokens=True, max_tokens=1, logits_processors=[mclp], logprobs=len(mclp.choices),)
  outputs = llm.generate(prompts, sampling_params_choice, use_tqdm=True)
  logprobs = [
    {lp.decoded_token: lp.logprob for lp in list(lps)}
    for lps in [output.outputs[0].logprobs[0].values() for output in outputs]
  ]
  choices = [max(d, key=d.get) for d in logprobs]
  print('generation end')
  df = df.with_columns(pl.Series('logprobs', logprobs), pl.Series('type', choices))
  print(df.group_by('type').agg(pl.len()).sort(['type']))
  logprobs = df['logprobs'].to_numpy()
  probs = softmax(logprobs, axis=-1)
  sub = df.with_columns(pl.Series('rule_violation', probs[:,0].tolist()))
  sub.select('row_id', 'rule_violation').write_csv('submission.csv')
  
  if not IS_SUB:
    import pandas as pd
    from sklearn.metrics import roc_auc_score
    
    # Load CSVs
    submission = pd.read_csv("submission.csv")
    gt = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/train.csv")[['row_id', 'rule_violation']]
    
    # Merge on 'row_id' to align predictions and ground truth
    merged = pd.merge(gt, submission, on="row_id", suffixes=('_gt', '_pred'))
    
    # Ensure proper columns
    y_true = merged['rule_violation_gt']
    y_score = merged['rule_violation_pred']
    
    # Compute AUC
    try:
        auc = roc_auc_score(y_true, y_score)
        print(f"Column-Averaged AUC with {N_PAIRS} pairs: {auc:.6f}")
    except ValueError as e:
        print(f"Cannot compute AUC: {e}")
        
  print(f"✅ Completed inference with N_PAIRS = {N_PAIRS}")

In [ ]:
%%writefile prepare_basicllm_semantic_data.py
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
import random
import numpy as np
import os
random.seed(42)
np.random.seed(42)
IS_SUB=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))


# Constants
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"
EMBEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"
TOP_K_SEMANTIC = 100  # Increased for more pairs
EMBEDDING_BATCH_SIZE = 128
N_PAIRS = 3  # Number of positive/negative example pairs (configurable)


def build_labeled_corpus(data_path):
    """Build comprehensive labeled corpus from all available data"""
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    corpus = []

    # Add train data
    for _, row in train_dataset.iterrows():
        corpus.append({
            "body": row["body"],
            "rule": row["rule"],
            "subreddit": row["subreddit"],
            "rule_violation": row["rule_violation"]
        })

    # Add positive examples from test data
    for _, row in test_dataset.iterrows():
        for i in [1, 2]:
            corpus.append({
                "body": row[f"positive_example_{i}"],
                "rule": row["rule"],
                "subreddit": row["subreddit"],
                "rule_violation": 1
            })

    # Add negative examples from test data
    for _, row in test_dataset.iterrows():
        for i in [1, 2]:
            corpus.append({
                "body": row[f"negative_example_{i}"],
                "rule": row["rule"],
                "subreddit": row["subreddit"],
                "rule_violation": 0
            })

    corpus_df = pd.DataFrame(corpus).drop_duplicates().reset_index(drop=True)
    corpus_df["corpus_id"] = corpus_df.index
    return corpus_df


def main():
    # Build corpus for semantic search
    print("Building labeled corpus...")
    corpus_df = build_labeled_corpus(DATA_PATH)
    
    # Load embedding model
    print("Loading embedding model...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_PATH, device="cuda")
    
    # Load test data
    if IS_SUB:
        test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    else:
        test_dataframe = pd.read_csv(f"{DATA_PATH}/train.csv")
        
    # Group test data by rule for efficient batch processing
    rule_groups = {}
    for _, row in test_dataframe.iterrows():
        rule = row["rule"]
        if rule not in rule_groups:
            rule_groups[rule] = []
        rule_groups[rule].append(row.to_dict())
    
    print(f"Processing {len(test_dataframe)} test examples across {len(rule_groups)} rules")
    print(f"Using N_PAIRS = {N_PAIRS} (collecting {N_PAIRS} positive + {N_PAIRS} negative examples)")
    
    all_test_data = []
    
    # Process each rule group with batch optimization
    for rule, test_rows in tqdm(rule_groups.items(), desc="Processing rule groups"):
        # Filter corpus by rule only (not subreddit for larger corpus)
        rule_corpus = corpus_df[corpus_df["rule"] == rule].reset_index(drop=True)
        
        if len(rule_corpus) == 0:
            # Fallback to entire corpus
            rule_corpus = corpus_df.reset_index(drop=True)
        
        # Pre-encode corpus once for this rule
        print(f"Encoding corpus for {rule[:50]}... ({len(rule_corpus)} examples)")
        corpus_embeddings = embedding_model.encode(
            rule_corpus["body"].tolist(),
            batch_size=EMBEDDING_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        
        # Extract all test bodies for batch encoding
        test_bodies = [row["body"] for row in test_rows]
        
        print(f"Encoding {len(test_bodies)} test comments...")
        test_embeddings = embedding_model.encode(
            test_bodies,
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=EMBEDDING_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        
        # Batch semantic search for all test examples in this rule
        search_results_batch = semantic_search(
            test_embeddings,
            corpus_embeddings,
            top_k=min(TOP_K_SEMANTIC, len(rule_corpus)),
            score_function=dot_score,
        )
        
        # Pre-filter corpus by violation type for faster lookup
        positive_corpus = rule_corpus[rule_corpus["rule_violation"] == 1].reset_index(drop=True)
        negative_corpus = rule_corpus[rule_corpus["rule_violation"] == 0].reset_index(drop=True)
        
        # Process results for each test example
        for i, test_row in enumerate(test_rows):
            search_results = search_results_batch[i]
            
            # Collect examples efficiently
            positive_examples = []
            negative_examples = []
            used_bodies = {test_row["body"]}  # Track used bodies to avoid duplicates
            
            for result in search_results:
                corpus_idx = result["corpus_id"]
                example_row = rule_corpus.iloc[corpus_idx]
                body = example_row["body"]
                
                # Skip if already used or exact match
                if body in used_bodies:
                    continue
                
                # Add to appropriate list
                if example_row["rule_violation"] == 1 and len(positive_examples) < N_PAIRS:
                    positive_examples.append(body)
                    used_bodies.add(body)
                elif example_row["rule_violation"] == 0 and len(negative_examples) < N_PAIRS:
                    negative_examples.append(body)
                    used_bodies.add(body)
                
                # Early exit when we have enough
                if len(positive_examples) >= N_PAIRS and len(negative_examples) >= N_PAIRS:
                    break
            
            # Fast fallback using pre-filtered corpus
            while len(positive_examples) < N_PAIRS and len(positive_corpus) > 0:
                candidates = positive_corpus[~positive_corpus["body"].isin(used_bodies)]
                if len(candidates) > 0:
                    sample = candidates.sample(1)["body"].iloc[0]
                    positive_examples.append(sample)
                    used_bodies.add(sample)
                else:
                    positive_examples.append("No positive example found")
            
            while len(negative_examples) < N_PAIRS and len(negative_corpus) > 0:
                candidates = negative_corpus[~negative_corpus["body"].isin(used_bodies)]
                if len(candidates) > 0:
                    sample = candidates.sample(1)["body"].iloc[0]
                    negative_examples.append(sample)
                    used_bodies.add(sample)
                else:
                    negative_examples.append("No negative example found")
            
            # Ensure we have exactly N_PAIRS of each
            while len(positive_examples) < N_PAIRS:
                positive_examples.append("No positive example found")
            while len(negative_examples) < N_PAIRS:
                negative_examples.append("No negative example found")
            
            # Add semantic examples to test row
            test_result = test_row.copy()
            for j in range(N_PAIRS):
                test_result[f"semantic_positive_example_{j+1}"] = positive_examples[j]
                test_result[f"semantic_negative_example_{j+1}"] = negative_examples[j]
            
            all_test_data.append(test_result)
        
        # Clear embeddings to save memory
        del corpus_embeddings, test_embeddings
        torch.cuda.empty_cache()
    
    # Save results
    result_df = pd.DataFrame(all_test_data)
    result_df.to_csv("test_with_semantic_examples_basicllm.csv", index=False)
    print(f"✅ Saved test_with_semantic_examples_basicllm.csv with {len(result_df)} examples")
    
    # Clear GPU memory
    del embedding_model
    torch.cuda.empty_cache()
    
    # Show a sample
    print(f"\nSample semantic examples (N_PAIRS = {N_PAIRS}):")
    sample = result_df.iloc[0]
    print(f"Rule: {sample['rule']}")
    print(f"Target: {sample['body'][:100]}...")
    for j in range(N_PAIRS):
        print(f"Semantic Pos {j+1}: {sample[f'semantic_positive_example_{j+1}'][:100]}...")
        print(f"Semantic Neg {j+1}: {sample[f'semantic_negative_example_{j+1}'][:100]}...")


if __name__ == "__main__":
    main()

In [3]:
 %%bash
# # WHEELHOUSE=/kaggle/input/mdc-wheelhouse/wheelhouse
uv pip uninstall --system 'tensorflow'
uv pip install  'polars==1.31.0' 'vllm==0.10.0' 'logits-processor-zoo==0.2.1'
uv pip install 'triton==3.2.0'

Using Python 3.11.13 environment at: /usr
Uninstalled 1 package in 4.67s
 - tensorflow==2.18.0
Using Python 3.11.13 environment at: /usr
Resolved 151 packages in 1.51s
 Downloaded nvidia-cufile-cu12
 Downloaded outlines-core
 Downloaded tokenizers
 Downloaded torchaudio
 Downloaded uvloop
 Downloaded numba
 Downloaded pycountry
 Downloaded mistral-common
 Downloaded torchvision
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded sympy
 Downloaded xgrammar
 Downloaded llguidance
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded transformers
 Downloaded polars
 Downloaded llvmlite
 Downloaded nvidia-curand-cu12
 Downloaded xformers
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded triton
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded vllm
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 58 packages in 42.23s
Uninstalled 26 packages in 3.1

In [4]:
!python prepare_basicllm_semantic_data.py

Building labeled corpus...
Loading embedding model...
Processing 2029 test examples across 2 rules
Processing rule groups:   0%|                             | 0/2 [00:00<?, ?it/s]Encoding corpus for No Advertising: Spam, referral links, unsolicited ... (963 examples)
Encoding 1012 test comments...
Processing rule groups:  50%|██████████▌          | 1/2 [00:47<00:47, 47.60s/it]Encoding corpus for No legal advice: Do not offer or request legal adv... (1020 examples)
Encoding 1017 test comments...
Processing rule groups: 100%|█████████████████████| 2/2 [01:31<00:00, 45.74s/it]
✅ Saved test_with_semantic_examples_basicllm.csv with 2029 examples

Sample semantic examples:
Rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.
Target: Banks don't want you to know this! Click here to know more!...
Semantic Pos 1: Sounds like you need http://understandingrelationships.com

I'm not trolling, I follow his material ...
Semantic Neg 1: There's

In [ ]:
! VLLM_USE_V1=0 TORCH_CUDA_ARCH_LIST=7.5 python check_semantic.py

importing vllm...
loading vllm...
INFO 09-27 10:16:14 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 09-27 10:16:26 [config.py:1604] Using max model len 4096
WARNING 09-27 10:16:27 [config.py:1084] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 09-27 10:16:28 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 09-27 10:16:28 [llm_engine.py:228] Initializing a V0 LLM engine (v0.10.0) with config: model='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', speculative_config=None, tokenizer='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUT

In [ ]:
! head -n 5 submission.csv